# 08 · Optuna HPO

Tune LightGBM on **train + validation only**. The Aug 5–21 holdout is never read here.

Start with ~30 trials on the M4 Air and raise if runtime allows.

In [ ]:
from cross_model_drift.data import load_split
from cross_model_drift.features import target_vector
from cross_model_drift.hpo import HPO_PARAM_SPACE, run_optuna_hpo
from cross_model_drift.metrics import quality_metrics
from cross_model_drift.models import train_lightgbm
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.tracking import clearml_task, log_metrics, log_parameters

nb = setup_model_session()
N_TRIALS = 8  # raise to 30–50 for the full PoC
train = load_split("v1", "train", nb.config, engine=nb.engine)
valid = load_split("v1", "validation", nb.config, engine=nb.engine)
test = load_split("v1", "test", nb.config, engine=nb.engine)

In [ ]:
study = run_optuna_hpo(train, valid, n_trials=N_TRIALS, threshold=nb.threshold)
study.best_value, study.best_params

In [ ]:
best = train_lightgbm(
    train,
    target_vector(train),
    valid,
    target_vector(valid),
    params=study.best_params,
    threshold=nb.threshold,
)
test_metrics = quality_metrics(target_vector(test), best.predict_proba(test), threshold=nb.threshold)
path = best.save(nb.artifacts / "models" / "v1_lightgbm_tuned.joblib")
trials = study.trials_dataframe()
trials.to_parquet(nb.artifacts / "hpo" / "v1_optuna_trials.parquet", index=False)
with clearml_task("run_optuna_hpo", config=nb.config, task_type="optimizer", tags=["v1", "optuna"], init=True) as task:
    log_parameters(task, study.best_params, name="best_params")
    log_metrics(task, test_metrics, title="v1_test")
path, test_metrics

In [ ]:
HPO_PARAM_SPACE, trials[["number", "value", "state"]].head()